# woodelf vs. shapiq TreeSHAP-IQ — Census Income Benchmark

This notebook benchmarks **woodelf** (`WoodelfExplainer`) against
**shapiq**'s `TreeExplainer` (which implements the TreeSHAP-IQ algorithm,
Muschalik et al., AAAI 2024) on the UCI Adult / Census Income dataset.

**woodelf** ("From Decision Trees to Boolean Logic: A Fast and Unified SHAP
Algorithm", AAAI 2026) is a Python package that implements a vectorised,
NumPy/SciPy-based SHAP algorithm for tree ensembles. Its API is a drop-in
replacement for `shap.TreeExplainer`.

## What we measure

| Quantity | woodelf | shapiq |
|---|---|---|
| Shapley values (SV) | `WoodelfExplainer(model).shap_values(X)` | `max_order=1, index="SV"` |
| Pairwise interactions | `WoodelfExplainer(model).shap_interaction_values(X)` | `max_order=2, index="k-SII"` |

A few notes on the comparison:

* **For order-1 Shapley values**, woodelf (path-dependent mode, no background
  data) and shapiq should agree numerically with the standard SV up to
  floating-point noise. We sanity-check this.
* **For interactions, the definitions differ.** woodelf returns the TreeSHAP
  interaction values of Lundberg et al. (2020), exactly matching
  `shap.TreeExplainer.shap_interaction_values`. shapiq returns k-SII values
  (Bordt & von Luxburg, 2023), which aggregate higher-order terms differently.
  At order 2 the compute cost is essentially the same for both, so the timing
  comparison is still informative — just note the values are *not* expected to
  match numerically.
* **Perturbation mode**: both libraries default to the tree-path-dependent
  perturbation (no background dataset). woodelf switches to interventional
  (Background) SHAP when a background dataset is passed to the constructor;
  see the Caveats section.

## Parallelism

* **shapiq** uses joblib: `shapiq.TreeExplainer(...).explain_X(X, n_jobs=-1)`.
* **woodelf** uses NumPy/SciPy vectorisation internally and does **not** expose
  an `n_jobs` parameter; it parallelises automatically via NumPy's thread pool
  (BLAS / OpenMP). Both are run on all available cores.

## References

* woodelf paper: [arXiv:2511.09376](https://arxiv.org/abs/2511.09376)
* shapiq / TreeSHAP-IQ: Muschalik et al., AAAI 2024
* TreeSHAP interactions: Lundberg et al., Nature Machine Intelligence, 2020

## 1. Setup

In [16]:
# Install (uncomment if needed)
# %pip install woodelf-explainer shapiq xgboost lightgbm scikit-learn pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb

from woodelf import WoodelfExplainer
import shapiq

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

import woodelf
print("woodelf:  ", woodelf.__version__ if hasattr(woodelf, "__version__") else "?")
print("shapiq:   ", shapiq.__version__)

woodelf:   0.4.2
shapiq:    None


## 2. Load and pre-process Census Income data

We fetch the Adult dataset from OpenML (id 1590, the standard "adult" task)
instead of relying on local files. The pre-processing mirrors the original
notebook: one-hot encode the categorical columns, drop missing-value markers,
and keep the 13 informative columns.

In [18]:
# Fetch the Adult / Census Income dataset
adult = fetch_openml("adult", version=2, as_frame=True)
X_raw = adult.data
y_raw = adult.target

# Encode label: ">50K" -> 1, "<=50K" -> 0
y = (y_raw == ">50K").astype(int).values

categorical_cols = X_raw.select_dtypes(include=["category", "object"]).columns.tolist()

# Replace "?" markers with NaN, then drop those rows
X_clean = X_raw.copy()
for col in categorical_cols:
    X_clean[col] = X_clean[col].astype(str).replace("?", np.nan)
mask = X_clean.notna().all(axis=1)
X_clean = X_clean[mask].reset_index(drop=True)
y = y[mask.values]

# One-hot encode categoricals
X = pd.get_dummies(X_clean, columns=categorical_cols, drop_first=False)
# Cast bool dummies to int so xgboost / lightgbm are happy
X = X.astype({c: np.int8 for c in X.columns if X[c].dtype == bool})

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

print(f"Training data: {X_train.shape[0]} rows × {X_train.shape[1]} columns")
print(f"Testing  data: {X_test.shape[0]} rows × {X_test.shape[1]} columns")
print(f"Positive class share (train): {y_train.mean():.3f}")

Training data: 33916 rows × 104 columns
Testing  data: 11306 rows × 104 columns
Positive class share (train): 0.248


## 3. Helper functions

* `run_woodelf` — creates a `WoodelfExplainer` (path-dependent mode, no
  background data), runs `num_round` times and reports mean ± std wall-clock.
  Set `interactions=True` to call `.shap_interaction_values()` instead of
  `.shap_values()`.
* `run_shapiq` — same pattern for shapiq's `TreeExplainer`.
* `woodelf_sv_to_array` — normalises the woodelf output (which mirrors
  `shap.TreeExplainer`) to a plain `(n_samples, n_features)` array so we can
  compare against shapiq's order-1 values.

In [41]:
def run_woodelf(model, sample, *, interactions, num_round, num_sample,
                background=None):
    """
    Time WoodelfExplainer `num_round` times. Returns the last result.

    Parameters
    ----------
    model       : trained tree model (RF / XGBoost / LightGBM)
    sample      : DataFrame to explain (first `num_sample` rows are used)
    interactions: if True, call shap_interaction_values() instead of shap_values()
    num_round   : number of timing repetitions
    num_sample  : number of samples to explain per round
    background  : optional background dataset (enables interventional SHAP);
                  pass None (default) for path-dependent mode
    """
    explainer = WoodelfExplainer(model, background)
    # Pass DataFrame (not .values) — woodelf needs .columns to load the model
    X_sub = sample.iloc[:num_sample]
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        if interactions:
            out = explainer.shap_interaction_values(X_sub)
        else:
            out = explainer.shap_values(X_sub)
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    mode = "background" if background is not None else "path-dependent"
    label = f"woodelf ({mode})" + (" interactions" if interactions else "")
    print(f"=> {label}: {run_time.mean():.3f} ± {run_time.std():.3f} s")
    return out, run_time

In [20]:
def run_shapiq(model, sample, *, max_order, n_jobs, num_round, num_sample,
               index=None, class_index=None):
    """Time shapiq's TreeExplainer `num_round` times. Returns the last result."""
    if index is None:
        index = "SV" if max_order == 1 else "k-SII"
    explainer = shapiq.TreeExplainer(
        model=model,
        max_order=max_order,
        min_order=1,
        index=index,
        class_index=class_index,
    )
    X_arr = sample.iloc[:num_sample].values
    run_time = np.zeros(num_round)
    out = None
    for i in range(num_round):
        start = time.time()
        out = explainer.explain_X(X_arr, n_jobs=n_jobs)
        run_time[i] = time.time() - start
        print(f"  Round {i + 1}: {run_time[i]:.3f} s")
    label = f"shapiq (max_order={max_order}, index={index})"
    print(f"=> {label}: {run_time.mean():.3f} ± {run_time.std():.3f} s")
    return out, run_time

In [21]:
def woodelf_sv_to_array(welf_out, class_index=None):
    """
    Normalise woodelf / shap.TreeExplainer SV output to (n_samples, n_features).

    woodelf mirrors shap.TreeExplainer output:
    - Binary RF / LightGBM (multi-output): list of 2 arrays, each
      (n_samples, n_features). We take element `class_index` (default 1).
    - XGBoost binary (single log-odds output): 2-D array (n_samples, n_features).
    """
    if isinstance(welf_out, list):
        idx = class_index if class_index is not None else 1
        return np.asarray(welf_out[idx])
    return np.asarray(welf_out)


def shapiq_sv_to_array(iv_result, n_features):
    """Convert shapiq's per-sample InteractionValues (order 1) to (n_samples, n_features)."""
    if isinstance(iv_result, list):
        rows = []
        for iv in iv_result:
            row = iv.get_n_order_values(1)
            rows.append(np.asarray(row).ravel())
        return np.vstack(rows)
    return np.asarray(iv_result.values)

In [22]:
# Global benchmark settings
NUM_SAMPLE_SV  = 1000   # samples to explain for SV benchmark
NUM_SAMPLE_INT = 100    # samples to explain for interaction benchmark
NUM_ROUND      = 3      # repetitions for mean/std
N_JOBS         = -1     # shapiq: all available cores

## 4. Random Forest (scikit-learn)

`RandomForestClassifier(n_estimators=200, max_depth=8)` — same configuration
as the FastTreeSHAP benchmark notebook.

In [23]:
n_estimators = 200
max_depth    = 8

rf_model = RandomForestClassifier(
    n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=0
)
rf_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, rf_model.predict(X_test)):.3f}")

AUC:      0.904
Accuracy: 0.848


### 4a. Shapley values — correctness check

For path-dependent SHAP at order 1, woodelf and shapiq (index=`"SV"`) both
implement the standard Shapley value and should agree up to floating-point noise.

In [42]:
# woodelf (path-dependent, no background data)
sv_welf, _ = run_woodelf(
    rf_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)

# shapiq (max_order=1, index="SV", class_index=1 for positive class)
sv_shapiq, _ = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:03<00:00, 56.37it/s]


M time: 0.03 sec, s time: 1.22 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.37648940086364746, f mean non zero size: 177.58837332525925)
  Round 1: 7.035 s
=> woodelf (path-dependent): 7.035 ± 0.000 s
  Round 1: 1317.833 s
=> shapiq (max_order=1, index=SV): 1317.833 ± 0.000 s


In [43]:
# Compare SV values
# woodelf for RF binary returns a list of 2 arrays; take index 1 (positive class)
sv_welf_arr   = woodelf_sv_to_array(sv_welf, class_index=1)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])

print(f"woodelf vs shapiq: max |diff| = {np.max(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")
print(f"                   mean |diff|= {np.mean(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")

woodelf vs shapiq: max |diff| = 8.72e-01
                   mean |diff|= 6.90e-03


### 4b. Timing benchmark — Shapley values

Run each method `NUM_ROUND` times and report mean ± std.

In [44]:
print("--- woodelf (path-dependent SV) ---")
_, t_rf_welf_sv = run_woodelf(
    rf_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_rf_sq_sv = run_shapiq(
    rf_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

--- woodelf (path-dependent SV) ---


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:03<00:00, 55.78it/s]


M time: 0.0 sec, s time: 1.23 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.3809192180633545, f mean non zero size: 177.58837332525925)
  Round 1: 3.662 s


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:03<00:00, 52.19it/s]


M time: 0.0 sec, s time: 1.32 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.40743494033813477, f mean non zero size: 177.58837332525925)
  Round 2: 3.835 s


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:03<00:00, 52.71it/s]


M time: 0.0 sec, s time: 1.3 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.40433835983276367, f mean non zero size: 177.58837332525925)
  Round 3: 3.797 s
=> woodelf (path-dependent): 3.765 ± 0.074 s

--- shapiq (SV) ---
  Round 1: 1956.729 s
  Round 2: 8093.423 s
  Round 3: 1364.730 s
=> shapiq (max_order=1, index=SV): 3804.961 ± 3042.017 s


### 4c. Pairwise interactions — woodelf vs shapiq

**Important**: the two libraries use different interaction indices.
woodelf returns the **TreeSHAP interaction values** (Lundberg et al. 2020),
which produce an $n \times n$ matrix with main effects on the diagonal and
symmetrically split pairwise interactions. shapiq returns **k-SII** values,
which aggregate higher-order terms differently. The values are not numerically
equal, but the timing comparison remains valid.

Sample count is reduced from `NUM_SAMPLE_SV` to `NUM_SAMPLE_INT` because
interaction values cost $O(n_{\text{features}})$ more per sample.

In [45]:
print("--- woodelf (TreeSHAP interactions) ---")
_, t_rf_welf_int = run_woodelf(
    rf_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_rf_sq_int = run_shapiq(
    rf_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
    class_index=1,
)

--- woodelf (TreeSHAP interactions) ---


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:04<00:00, 42.21it/s]


M time: 0.01 sec, s time: 1.86 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.424793004989624, f mean non zero size: 177.58837332525925)
Compute also shapley values in order to find the interactions of features with themselves. 
            The interaction of a feature with itself is its shapley value minus all the shapley 
            interaction values it has with other features (when it is the first feature in the pair).
            a.k.a:
            shap_(i,i) = shap_i - \sum_(j!=i) shap_(i,j) 


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:04<00:00, 49.20it/s]


M time: 0.0 sec, s time: 1.49 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.4595808982849121, f mean non zero size: 177.58837332525925)
  Round 1: 9.070 s


...

M time: 0.0 sec, s time: 1.48 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.45403146743774414, f mean non zero size: 177.58837332525925)
  Round 2: 8.944 s


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:04<00:00, 45.68it/s]


M time: 0.0 sec, s time: 1.72 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.3943610191345215, f mean non zero size: 177.58837332525925)
Compute also shapley values in order to find the interactions of features with themselves. 
            The interaction of a feature with itself is its shapley value minus all the shapley 
            interaction values it has with other features (when it is the first feature in the pair).
            a.k.a:
            shap_(i,i) = shap_i - \sum_(j!=i) shap_(i,j) 


Preprocessing the trees and computing SHAP: 100%|██████████| 200/200 [00:03<00:00, 56.73it/s]


M time: 0.0 sec, s time: 1.29 sec (26422 _get_s_vectors_given_f calls, f prepare time: 0.39516210556030273, f mean non zero size: 177.58837332525925)
  Round 3: 8.063 s
=> woodelf (path-dependent) interactions: 8.692 ± 0.448 s

--- shapiq (k-SII, max_order=2) ---
  Round 1: 385.582 s
  Round 2: 377.767 s
  Round 3: 371.385 s
=> shapiq (max_order=2, index=k-SII): 378.245 ± 5.806 s


## 5. XGBoost

`XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1)`.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    eval_metric="logloss", random_state=0,
)
xgb_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, xgb_model.predict(X_test)):.3f}")

### 5a. Shapley values — correctness check

In [ ]:
# XGBoost binary classifier: woodelf returns a single 2-D array (n_samples, n_features)
# (log-odds space). shapiq with class_index=None also gives a single output.
sv_welf, _ = run_woodelf(
    xgb_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
)

sv_welf_arr   = woodelf_sv_to_array(sv_welf)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])
print(f"woodelf vs shapiq: max |diff| = {np.max(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")
print(f"                   mean |diff|= {np.mean(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")

### 5b. Timing — Shapley values

In [ ]:
print("--- woodelf (path-dependent SV) ---")
_, t_xgb_welf_sv = run_woodelf(
    xgb_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_xgb_sq_sv = run_shapiq(
    xgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)

### 5c. Timing — pairwise interactions

In [ ]:
print("--- woodelf (TreeSHAP interactions) ---")
_, t_xgb_welf_int = run_woodelf(
    xgb_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_xgb_sq_int = run_shapiq(
    xgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 6. LightGBM

`LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.1)`.

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1,
    random_state=0, verbosity=-1,
)
lgb_model.fit(X_train, y_train)

print(f"AUC:      {roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1]):.3f}")
print(f"Accuracy: {accuracy_score(y_test, lgb_model.predict(X_test)):.3f}")

### 6a. Shapley values — correctness check

In [ ]:
# LightGBM binary classifier: woodelf returns a list of 2 arrays; take index 1
sv_welf, _ = run_woodelf(
    lgb_model, X_test, interactions=False,
    num_round=1, num_sample=NUM_SAMPLE_SV,
)
sv_shapiq, _ = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=1, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

sv_welf_arr   = woodelf_sv_to_array(sv_welf, class_index=1)
sv_shapiq_arr = shapiq_sv_to_array(sv_shapiq, X_test.shape[1])
print(f"woodelf vs shapiq: max |diff| = {np.max(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")
print(f"                   mean |diff|= {np.mean(np.abs(sv_welf_arr - sv_shapiq_arr)):.2e}")

### 6b. Timing — Shapley values

In [ ]:
print("--- woodelf (path-dependent SV) ---")
_, t_lgb_welf_sv = run_woodelf(
    lgb_model, X_test, interactions=False,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
)
print()
print("--- shapiq (SV) ---")
_, t_lgb_sq_sv = run_shapiq(
    lgb_model, X_test, max_order=1,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_SV,
    class_index=1,
)

### 6c. Timing — pairwise interactions

In [ ]:
print("--- woodelf (TreeSHAP interactions) ---")
_, t_lgb_welf_int = run_woodelf(
    lgb_model, X_test, interactions=True,
    num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)
print()
print("--- shapiq (k-SII, max_order=2) ---")
_, t_lgb_sq_int = run_shapiq(
    lgb_model, X_test, max_order=2,
    n_jobs=N_JOBS, num_round=NUM_ROUND, num_sample=NUM_SAMPLE_INT,
)

## 7. Summary table

Aggregate all timing results. `SV` rows are over `NUM_SAMPLE_SV` samples;
`Interaction` rows are over `NUM_SAMPLE_INT` samples.

In [ ]:
def fmt(t):
    return f"{t.mean():.3f} ± {t.std():.3f}"

summary = pd.DataFrame(
    {
        "RF": [
            fmt(t_rf_welf_sv),  fmt(t_rf_sq_sv),
            fmt(t_rf_welf_int), fmt(t_rf_sq_int),
        ],
        "XGBoost": [
            fmt(t_xgb_welf_sv),  fmt(t_xgb_sq_sv),
            fmt(t_xgb_welf_int), fmt(t_xgb_sq_int),
        ],
        "LightGBM": [
            fmt(t_lgb_welf_sv),  fmt(t_lgb_sq_sv),
            fmt(t_lgb_welf_int), fmt(t_lgb_sq_int),
        ],
    },
    index=pd.MultiIndex.from_tuples(
        [
            ("SV",          f"woodelf path-dep  ({NUM_SAMPLE_SV} samples)"),
            ("SV",          f"shapiq SV         ({NUM_SAMPLE_SV} samples)"),
            ("Interaction", f"woodelf TreeSHAP  ({NUM_SAMPLE_INT} samples)"),
            ("Interaction", f"shapiq k-SII      ({NUM_SAMPLE_INT} samples)"),
        ],
        names=["Quantity", "Method"],
    ),
)
summary

## 8. Caveats and things to try

* **Perturbation mode**: the timings above use path-dependent perturbation
  (no background data), matching shapiq's default. To benchmark woodelf's
  interventional / Background SHAP mode, pass `background=X_train.values`
  to `run_woodelf`. Note that interventional SHAP requires computing over the
  background set at explanation time, so it is typically slower — but woodelf
  is specifically optimised for this case (see the paper).

* **GPU acceleration**: woodelf supports GPU via CuPy. Replace
  `WoodelfExplainer(model, background)` with
  `WoodelfExplainer(model, background, GPU=True)` to offload to the GPU.
  shapiq has no GPU support.

* **Interaction index choice for shapiq**: we used `k-SII`. To benchmark
  `SII` (mathematically closest to TreeSHAP's interaction matrix) or `STII`,
  pass `index="SII"` or `index="STII"` to `run_shapiq`. Compute cost at
  `max_order=2` is essentially the same.

* **Higher-order interactions**: shapiq supports order-3+ interactions
  (TreeSHAP-IQ's main selling point). woodelf also supports interactions
  but its `shap_interaction_values` returns the standard pairwise TreeSHAP
  matrix. Bump `max_order=3` in a `run_shapiq` call to see what that costs.

* **Banzhaf values**: woodelf additionally provides `explainer.banzhaf_values(X)`
  and `explainer.banzhaf_interaction_values(X)`, which shapiq also supports
  via `index="BV"` and `index="BII"`.

* **Sample sizes**: `NUM_SAMPLE_SV` was set to `1000` because shapiq's
  TreeSHAP-IQ is typically slower per sample. Crank it up for longer-running
  comparisons.